In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import json
sys.path.append(os.path.dirname(os.path.abspath('.')))
from utils.load_api_keys import get_env_var
from pathlib import Path
from pprint import pprint

import glob

In [2]:
HF_TOKEN = get_env_var("HF_TOKEN")
GROQ_API_KEY = get_env_var("GROQ_API_KEY")
FILE_PATH_MAJORS = get_env_var("FILE_PATH_MAJORS")
PERSIST_DIR = get_env_var("PERSIST_DIR")
PERSIST_DIR_FAISS = get_env_var("PERSIST_DIR_FAISS")
OPENAI_API_KEY = get_env_var("OPENAI_API_KEY")
FILE_PATH_MINORS = get_env_var("FILE_PATH_MINORS")
FILE_PATH_ADDMAJORS = get_env_var("FILE_PATH_ADDMAJORS")
FILE_PATH_COURSES = get_env_var("FILE_PATH_COURSES")

In [3]:
from langchain_community.document_loaders import JSONLoader, MergedDataLoader
from langchain_text_splitters import RecursiveJsonSplitter, RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [4]:
#loading the embedding model and llm model 
embeddings1=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
model1 = ChatGroq(model="Gemma2-9b-It", groq_api_key =  GROQ_API_KEY)
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=OPENAI_API_KEY)
model = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)

In [47]:
major_files = ['Biological Sciences','Business Administration','Computer Science','GenEd', 'Information Systems']
programs = ['BS','BA','CS','GENED','IS']
json_major_files = {}
for f in major_files:
    json_major_files[f] = glob.glob(os.path.join(FILE_PATH_MAJORS+f+'/'+'*.json'))
    
json_Courses_files = glob.glob(os.path.join(FILE_PATH_COURSES,'*.json'))
json_minor_files = glob.glob(os.path.join(FILE_PATH_MINORS,'*.json'))
json_addmajor_files = glob.glob(os.path.join(FILE_PATH_ADDMAJORS,'*.json'))

In [48]:
def major_metadata_func(department, category):
    def metadata_func(record: dict, metadata: dict) -> dict:
        if "source" in metadata:
            metadata["source"] = Path(metadata["source"]).stem
        metadata["department"] = department  
        metadata["category"] = category
        return metadata
    return metadata_func

In [49]:
Major_docs = {}
i =0
for file,v in json_major_files.items():
    li = []
    for val in v:
        loader = JSONLoader(file_path=val,
                            jq_schema=".[]",
                            text_content=False,
                            metadata_func=major_metadata_func(programs[i],programs[i]+" Major"))
        raw_docs = loader.load()
        li.extend(raw_docs)
    Major_docs[programs[i]] = li
    i = i + 1

for k,v in Major_docs.items():
    print(k)
    print("length of docs:",len(v),v[0])

BS
length of docs: 99 page_content='{"Concentration Name": "Biochemistry Option", "Concentration Description": "The Biochemistry option provides advanced training in the chemical processes underlying biological systems, including metabolism, enzyme kinetics, and structural biochemistry. :contentReference[oaicite:0]{index=0}", "Career Paths": "Biochemist, Pharmaceutical Researcher, Molecular Biologist, Biotechnology Scientist, Drug Development Specialist", "Concentration Course Requirements Section A": ["03-231 Honors Biochemistry (or 03-232 Biochemistry I) (9 units)", "03-344 Experimental Biochemistry (12 units)"], "Concentration Course Requirements Section B": ["Choose two of the following (9\u201312 units each):", "03-740 Advanced Biochemistry (12 units)", "03-442 Molecular Biology (9 units)", "03-435 Cancer Biology (9 units)"], "Summative Course": "No summative course needed to complete this concentration as a Biological Sciences major.", "Concentration Contact Personnel": "Your ind

In [51]:

def metadata_func_Courses(record: dict, metadata: dict) -> dict:
    if "source" in metadata:
        metadata["source"] = Path(metadata["source"]).stem
    metadata["departmentid"] = Path(metadata["source"]).stem.split('-')[0]  
    metadata['category'] = 'Courses'
    return metadata

Courses_docs = []

for file in json_Courses_files:
    loader = JSONLoader(file_path=file,
                            jq_schema="""
                                    def clean:
                                        
                                        if type == "object" then
                                            with_entries(select(.value != null
                                                and (( ( (.value | type) != "object" and (.value | type) != "array") and .value != "" )
                                                or ( ((.value | type) == "object" or (.value | type) == "array") and ((.value | clean | length) > 0) ))))
                                        elif type == "array" then
                                            map(clean) | map(select(. != null and . != "" and ((type != "object" and type != "array") or (length > 0))))
                                        else
                                            .
                                        end;
                                    clean
                                    """,
                            text_content=False,
                            metadata_func=metadata_func_Courses)
    raw_docs = loader.load()
    Courses_docs.extend(raw_docs)
print("length of docs:",len(Courses_docs))
print(Courses_docs[0].metadata,"\n",Courses_docs[0].page_content)


length of docs: 4736
{'source': '02-201', 'seq_num': 1, 'departmentid': '02', 'category': 'Courses'} 
 {"code": "02-201", "name": "Programming for Scientists", "base_name": "Programming for Scientists", "units": 10, "min_units": 10, "max_units": 10, "short_name": "PRGRMMING SCIENTISTS", "is_topic": false, "offered_in_campuses": [1], "offerings": [{"campus_id": 1, "semesters": [{"semester": 2, "year": 2016}, {"semester": 1, "year": 2016}, {"semester": 2, "year": 2017}, {"semester": 2, "year": 2018}, {"semester": 2, "year": 2019}], "sub_semesters": []}], "long_desc": "Provides a practical introduction to programming for students with little or no prior programming experience who are interested in science. Fundamental scientific algorithms will be introduced, and extensive programming assignments will be based on analytical tasks that might be faced by scientists, such as parsing,  simulation, and optimization.  Principles of good software engineering will also be stressed. The course wil

In [65]:
def metadata_func_Minors(record: dict, metadata: dict) -> dict:
    if "source" in metadata:
        metadata["source"] = Path(metadata["source"]).stem
    metadata["department"] = 'Minors' 
    metadata['category'] = 'Minors'
    return metadata

Minor_docs = []

for file in json_minor_files:
    loader = JSONLoader(file_path=file,
                        jq_schema=".[]",
                        text_content=False,
                        metadata_func=metadata_func_Minors)
    raw_docs = loader.load()
    Minor_docs.extend(raw_docs)
print("length of docs:",len(Minor_docs))
print(Minor_docs[0].metadata,"\n",Minor_docs[0].page_content)

length of docs: 45
{'source': 'Advisors', 'seq_num': 1, 'department': 'Minors', 'category': 'Minors'} 
 {"Minor Name": "Arabic Studies", "Advisor": "Ezzohra Moufid & Lama Nassif"}


In [53]:
def metadata_func_addMajors(record: dict, metadata: dict) -> dict:
    if "source" in metadata:
        metadata["source"] = Path(metadata["source"]).stem
    metadata["department"] = 'AddMajors' 
    metadata['category'] = 'AddMajors'
    return metadata

AddMajor_docs = []

for file in json_addmajor_files:
    loader = JSONLoader(file_path=file,
                        jq_schema=".[]",
                        text_content=False,
                        metadata_func=metadata_func_addMajors)
    raw_docs = loader.load()
    AddMajor_docs.extend(raw_docs)
print("length of docs:",len(AddMajor_docs))
print(AddMajor_docs[0].metadata,"\n",AddMajor_docs[0].page_content)

length of docs: 3
{'source': 'Biological_Sciences_Additional_Major_Requirements', 'seq_num': 1, 'department': 'AddMajors', 'category': 'AddMajors'} 
 {"Minor Name": "Additional Major in Biological Sciences", "Official Requirements Reference": "Undergraduate Department of Biological Sciences", "Description": "The Additional Major in Biological Sciences offers a comprehensive curriculum spanning modern biology, genetics, biochemistry, computational biology, cell biology, and advanced experimental techniques, culminating in hands-on research experiences. Core coursework builds a solid foundation in molecular and cellular processes, quantitative genetic analysis, and computational approaches, while a structured chemistry sequence ensures mastery of analytical and synthetic methods. A broad elective component allows exploration of specialized topics at the 300-level or above, including independent and interdisciplinary research, with clear limits on research credit to maintain academic rigo

In [55]:
for k,docs in Major_docs.items():
    if k != "IS":
        print(k)

BS
BA
CS
GENED


In [ ]:
jsonsplitter = RecursiveJsonSplitter(max_chunk_size=1000)
for k,docs in Major_docs.items():
    Major_json_chunks = []
    for doc in docs:
        if isinstance(doc.page_content, str):
            content = json.loads(doc.page_content)
        else:
            content = doc.page_content
                
        chunks = jsonsplitter.split_text(content, convert_lists=True)
            
        for chunk in chunks:
            Major_json_chunks.append(Document(
                page_content=chunk,
                metadata=doc.metadata
            ))
    ISmajor_db=Chroma.from_documents(documents=Major_json_chunks,
                                     embedding=embeddings,
                                     persist_directory=PERSIST_DIR+k+"/MajorData")
    ISmajor_db.persist()

In [57]:
jsonsplitter = RecursiveJsonSplitter(max_chunk_size=1000)
# Create chunks with metadata
Courses_json_chunks = []
for doc in Courses_docs:
    if isinstance(doc.page_content, str):
        content = json.loads(doc.page_content)
    else:
        content = doc.page_content
        
    chunks = jsonsplitter.split_text(content, convert_lists=True)
    
    for chunk in chunks:
        Courses_json_chunks.append(Document(
            page_content=chunk,
            metadata=doc.metadata
        ))
for jc in Courses_json_chunks[:5]:
    print(jc.metadata)
    print(jc.page_content)
    print("-"*100)

{'source': '02-201', 'seq_num': 1, 'departmentid': '02', 'category': 'Courses'}
{"code": "02-201", "name": "Programming for Scientists", "base_name": "Programming for Scientists", "units": 10, "min_units": 10, "max_units": 10, "short_name": "PRGRMMING SCIENTISTS", "is_topic": false, "offered_in_campuses": {"0": 1}, "offerings": {"0": {"campus_id": 1, "semesters": {"0": {"semester": 2, "year": 2016}, "1": {"semester": 1, "year": 2016}, "2": {"semester": 2, "year": 2017}, "3": {"semester": 2, "year": 2018}, "4": {"semester": 2, "year": 2019}}, "sub_semesters": {}}}, "long_desc": "Provides a practical introduction to programming for students with little or no prior programming experience who are interested in science. Fundamental scientific algorithms will be introduced, and extensive programming assignments will be based on analytical tasks that might be faced by scientists, such as parsing,  simulation, and optimization.  Principles of good software engineering will also be stressed. Th

In [73]:
jsonsplitter = RecursiveJsonSplitter(max_chunk_size=1000)
# Create chunks with metadata
Minors_json_chunks = []
for doc in Minor_docs:
    if isinstance(doc.page_content, str):
        content = json.loads(doc.page_content)
    else:
        content = doc.page_content
        
    chunks = jsonsplitter.split_text(content, convert_lists=True)
    
    for chunk in chunks:
        Minors_json_chunks.append(Document(
            page_content=chunk,
            metadata=doc.metadata
        ))
for jc in Minors_json_chunks[:5]:
    print(jc.metadata)
    print(jc.page_content)
    print("-"*100)
Minors_db=Chroma.from_documents(documents=Minors_json_chunks,
                                    embedding=embeddings,
                                    persist_directory=PERSIST_DIR+"/MinorData")
Minors_db.persist()

{'source': 'Advisors', 'seq_num': 1, 'department': 'Minors', 'category': 'Minors'}
{"Minor Name": "Arabic Studies", "Advisor": "Ezzohra Moufid & Lama Nassif"}
----------------------------------------------------------------------------------------------------
{'source': 'Advisors', 'seq_num': 2, 'department': 'Minors', 'category': 'Minors'}
{"Minor Name": "Biological Sciences", "Advisor": "Mohamed Bouaouina"}
----------------------------------------------------------------------------------------------------
{'source': 'Advisors', 'seq_num': 3, 'department': 'Minors', 'category': 'Minors'}
{"Minor Name": "Business Administration", "Advisor": "Agustin Indaco"}
----------------------------------------------------------------------------------------------------
{'source': 'Advisors', 'seq_num': 4, 'department': 'Minors', 'category': 'Minors'}
{"Minor Name": "Computer Science", "Advisor": "Giselle Reis"}
--------------------------------------------------------------------------------------

C:\Users\dkurup\AppData\Local\Temp\ipykernel_35900\2096834663.py:24: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  Minors_db.persist()


In [74]:
Addmajor_json_chunks = []
for doc in AddMajor_docs:
    if isinstance(doc.page_content, str):
        content = json.loads(doc.page_content)
    else:
        content = doc.page_content
        
    chunks = jsonsplitter.split_text(content, convert_lists=True)
    
    for chunk in chunks:
        Addmajor_json_chunks.append(Document(
            page_content=chunk,
            metadata=doc.metadata
        ))
for jc in Addmajor_json_chunks[:5]:
    print(jc.metadata)
    print(jc.page_content)
    print("-"*100)
addmajors_db=Chroma.from_documents(documents=Addmajor_json_chunks,
                                     embedding=embeddings,
                                     persist_directory=PERSIST_DIR+"/AddMajorData")
addmajors_db.persist()

{'source': 'Biological_Sciences_Additional_Major_Requirements', 'seq_num': 1, 'department': 'AddMajors', 'category': 'AddMajors'}
{"Minor Name": "Additional Major in Biological Sciences", "Official Requirements Reference": "Undergraduate Department of Biological Sciences", "Description": "The Additional Major in Biological Sciences offers a comprehensive curriculum spanning modern biology, genetics, biochemistry, computational biology, cell biology, and advanced experimental techniques, culminating in hands-on research experiences. Core coursework builds a solid foundation in molecular and cellular processes, quantitative genetic analysis, and computational approaches, while a structured chemistry sequence ensures mastery of analytical and synthetic methods. A broad elective component allows exploration of specialized topics at the 300-level or above, including independent and interdisciplinary research, with clear limits on research credit to maintain academic rigor. Whether pursued a

In [ ]:
from tqdm import tqdm
def store_documents_in_chrome(
    documents: list[Document],
    persist_dir: str = "",
    embedding_model=None,
    batch_size: int = 100,
):
    
    """
    Store a list of LangChain Document objects in ChromaDB with batching.

    Args:
        documents : List of documents to embed and store.
        persist_dir : Directory to persist ChromaDB.
        embedding_model: Embedding model .
        batch_size : Number of documents to process per batch.
    """
    if embedding_model is None:
        embedding_model = embeddings

    vectordb = Chroma(
        persist_directory=persist_dir,
        embedding_function=embedding_model
    )

    for i in tqdm(range(0, len(documents), batch_size), desc="Storing documents in Chroma"):
        batch = documents[i:i + batch_size]
        vectordb.add_documents(batch)

    # Persist to disk
    vectordb.persist()

    print(f"Stored {len(documents)} documents in '{persist_dir}'.")

    return vectordb
courses_db=store_documents_in_chrome(Courses_json_chunks,persist_dir=PERSIST_DIR+"Courses")

############# End of the file

########## Below are some examples

In [ ]:
## Saving to the disk
#ISmajor_db=Chroma.from_documents(documents=ISMajor_json_chunks,embedding=embeddings,persist_directory=PERSIST_DIR+"IS/MajorData")
#db = Chroma.from_documents(json_chunks, embeddings)
#retriever_ISMAJOR = db.as_retriever()
##All course in the "Courses" folder
#db=Chroma.from_documents(documents=course_data,embedding=embeddings,persist_directory=PERSIST_DIR+"Courses_all")
#db = Chroma.from_documents(json_chunks, embeddings)
#retriever_COURSES = db.as_retriever()
#ISmajor_db=store_documents_in_chrome(ISMajor_json_chunks,persist_dir=PERSIST_DIR+"IS/MajorData")
#courses_db=store_documents_in_chrome(Courses_json_chunks,persist_dir=PERSIST_DIR+"Courses")

Storing documents in Chroma: 100%|██████████| 127/127 [05:46<00:00,  2.73s/it]

Stored 12691 documents in '../chroma_db/Courses'.


In [78]:
vector_store = Chroma(persist_directory=PERSIST_DIR+"/MinorData", embedding_function=embeddings)


In [79]:
vector_store

In [80]:
r1 = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":10,
        })
retrieved_docs = r1.invoke(
    "What are the minor requirements for Biological Sciences"
)
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)


--- Document 1 ---
Content: {"Minor Name": "Minor in Biological Sciences", "Official Requirements Reference": "Department of Biological Sciences Catalog", "Description": "Provides students with foundational education in the biological sciences. A minimum of six biological sciences courses (and two chemistry prerequisites) must be completed to fulfill the minor. Total units required for the minor: 73. :contentReference[oaicite:0]{index=0}", "Prerequisites": "09-105 Introduction to Modern Chemistry I (10 units); 09-217 Organic Chemistry I (9 units) :contentReference[oaicite:1]{index=1}", "Required Courses": "Six biological sciences courses. They are: 03-121 or 03-151, 03-220 or 03-221, 03-231 or 03-232, 03-320, 03-xxx, 03-3xx :contentReference[oaicite:2]{index=2}", "Notes": "At least two chemistry prerequisites must be completed before enrolling in any biological sciences minor course. Biological sciences offerings include courses in biochemistry, cell biology, genetics, microbiology, p

In [38]:
import re

def extract_course_code(query: str) -> str:
    match = re.search(r"(\d{2,3})[- ]?(\d{3})", query)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return None

In [ ]:
query = "67-373"
course_code = extract_course_code(query)
if course_code:
    r1 = courses_db.as_retriever(search_kwargs={
        "k":10,
            "filter": {
                "source": course_code
            }
        })
else:
    r1 = courses_db.as_retriever(search_kwargs={"k":10})
retrieved_docs = r1.invoke(
    query
)
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)


--- Document 1 ---
Content: {"code": "67-373", "name": "Information Systems Consulting Project", "base_name": "Information Systems Consulting Project", "units": 12, "min_units": 12, "max_units": 12, "short_name": "IS CONSULTING PROJ", "is_topic": false, "prereqs": {"text": "67-272 [] at least D", "req_obj": {"id": 7946985, "screen_name": "udrfmpkl34h8x22", "original_min_units": null, "min_units": null, "is_shared": false, "is_uni_req": false, "is_concentration": false, "default_concentration": false, "choices": {"0": {"id": 7076, "screen_name": "67-272", "constraints": {"0": {"type": "course", "type_string": "", "data": {"course": {"code": "67-272", "id": 7075, "name": "Application Design and Development", "units": 12}}, "id": 7075, "is_hidden": false}}, "course_parent_id": "79469850000000007076"}}, "constraints": {"0": {"type": "anyxof", "type_string": "Fulfill all of the following requirements", "data": {"x": 1, "is_and": true}, "id": 11003032, "is_hidden": false}}}}}
Metadata: {'so